<a href="https://colab.research.google.com/github/Traiba/aurora-siger/blob/main/analysis-aurora-siger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import time
import random
import pandas as pd
import os
from google.colab import files
from google.colab import userdata
import openai

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = openai.OpenAI()

# ─────────────────────────────────────────────
#  1. UPLOAD DO ARQUIVO
# ─────────────────────────────────────────────
def carregar_arquivo_local():
    print("Por favor, faça o upload do arquivo 'dataset_telemetria_marte.csv'")
    uploaded = files.upload()
    for nome_arquivo in uploaded.keys():
        print(f' Arquivo "{nome_arquivo}" carregado com sucesso!')
        return nome_arquivo
    return None

# ─────────────────────────────────────────────
#  2. FUNÇÕES DE EXIBIÇÃO E AUXILIARES
# ─────────────────────────────────────────────
def exibir_sucesso():
    print("\n  UM PEQUENO PASSO PARA UM HOMEM, UM SALTO GIGANTE PARA O BRASIL!")
    print("  ESTAÇÃO MARCIANA ALCANÇADA - POUSO REALIZADO EM MARTE COM SUCESSO.")
    print("  [MISSÃO AURORA-SIGER CONCLUÍDA!]")

def exibir_falha(motivos_falha):
    print("\n  MISSÃO ABORTADA: FALHAS DETECTADAS")
    for motivo in motivos_falha:
        print(f"  - {motivo}")
    print("  PROCEDIMENTOS DE EMERGÊNCIA ATIVADOS. CÁPSULA EJETADA.")
    print("  [FALHA NA JORNADA PARA MARTE]")

def realizar_analise_energetica(carga_percentual):
    capacidade_total = 1000  # kWh
    consumo_decolagem = 300  # kWh
    perdas_percentual = 0.05
    energia_disponivel = capacidade_total * (carga_percentual / 100)
    perdas = energia_disponivel * perdas_percentual
    energia_real = energia_disponivel - perdas
    energia_restante = energia_real - consumo_decolagem
    return {
        "disponivel": energia_disponivel,
        "perdas": perdas,
        "real": energia_real,
        "restante": energia_restante,
        "seguro": energia_restante > 0
    }

# ─────────────────────────────────────────────
#  3. LÓGICA DO MINI-GAME
# ─────────────────────────────────────────────
#  FUNÇÃO ASSISTENTE DE IA
# ─────────────────────────────────────────────
def assistente_ia_dilema(dilema, telemetria_atual):
    telemetria_resumida = {
        'energia': telemetria_atual.get('nivel_energia', 'N/A'),
        'integridade': telemetria_atual.get('integridade_estrutural', 'N/A'),
        'o2_global': o2_pct_global,
        'combustivel': telemetria_atual.get('nivel_combustivel', 'N/A'),
        'pressao_tanques': telemetria_atual.get('pressao_tanques_percentual', 'N/A')
    }

    prompt = f"""Você é um assistente de IA para uma missão espacial. Analise o dilema e a telemetria resumida para dar uma sugestão MUITO CURTA (uma frase). Não escolha uma opção.

Dilema: {dilema['pergunta']}
Opções: {dilema['opcoes'][0]} OU {dilema['opcoes'][1]}
Telemetria: {telemetria_resumida}

Sugestão:"""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "Você é um assistente de IA conciso para uma missão espacial. Responda em uma frase curta."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=40
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Erro ao consultar a IA: {e}"


# ─────────────────────────────────────────────
def iniciar_mini_game(telemetria_atual):
    print(f"\n{'='*20}  DILEMA NO ESPAÇO PROFUNDO (RUMO A MARTE) {'='*20}")

    telemetria_pos_minigame = telemetria_atual.copy()

    dilemas = [
        {
            "pergunta": "Uma tempestade solar massiva está vindo! O que fazer?",
            "opcoes": [
                "1- Ativar escudos magnéticos (Gasta muita energia, protege integridade)",
                "2- Manter curso e confiar na blindagem (Risco de danos estruturais, economiza energia)"
            ],
            "consequencia1": {"nivel_energia": -15, "integridade_estrutural": 0, "msg": "Escudos ativados. Energia drenada, mas integridade mantida."},
            "consequencia2": {"nivel_energia": 0, "integridade_estrutural": 1, "msg": "Radiação solar atingiu os sistemas! Danos estruturais detectados."}
        },
        {
            "pergunta": "O sistema de Oxigênio está com um pequeno vazamento. O que fazer?",
            "opcoes": [
                "1- Selar o setor de carga (Reduz O2 disponível, mas evita perda total)",
                "2- Enviar drone para reparo externo (Gasta bateria extra, mantém O2)"
            ],
            "consequencia1": {"o2_pct_global": -20, "msg": "Setor selado. O2 reduzido, mas vazamento contido."},
            "consequencia2": {"nivel_energia": -10, "msg": "Drone realizou o reparo. Sistemas de suporte gastaram bateria extra."}
        },
        {
            "pergunta": "Anomalia na pressão dos tanques de combustível. O que fazer?",
            "opcoes": [
                "1- Aliviar pressão manualmente (Risco de perda de combustível, mas estabiliza pressão)",
                "2- Manter pressão e monitorar (Risco de explosão, mas mantém combustível)"
            ],
            "consequencia1": {"pressao_tanques_percentual": 99.0, "nivel_combustivel": -5, "msg": "Pressão aliviada. Pequena perda de combustível, mas tanques estabilizados."},
            "consequencia2": {"pressao_tanques_percentual": 105.0, "msg": "Pressão instável. Risco de falha catastrófica."}
        },
        {
            "pergunta": "Interferência inesperada na comunicação com a Terra. O que fazer?",
            "opcoes": [
                "1- Aumentar potência do transmissor (Gasta energia, mas melhora comunicação)",
                "2- Tentar reconfigurar antenas (Demora, mas economiza energia)"
            ],
            "consequencia1": {"nivel_energia": -10, "velocidade_comunicacao": 10.0, "msg": "Potência aumentada. Comunicação restabelecida, mas energia reduzida."},
            "consequencia2": {"velocidade_comunicacao": 3.0, "msg": "Reconfiguração falhou. Comunicação crítica."}
        }
    ]
    dilema = random.choice(dilemas)
    print(f"\n {dilema['pergunta']}")
    for opt in dilema['opcoes']: print(f"   {opt}")

    print("\nCONSULTANDO ASSISTENTE DE IA...")
    sugestao_ia = assistente_ia_dilema(dilema, telemetria_atual)
    print(f"ASSISTENTE DE IA: {sugestao_ia}")


    while True:
        escolha = input("Sua escolha (1 ou 2): ")
        if escolha in ["1", "2"]:
            break
        else:
            print("Opção inválida. Por favor, digite 1 ou 2.")
    res = dilema["consequencia1"] if escolha == "1" else dilema["consequencia2"]
    print(f"\n RESULTADO: {res['msg']}")

    for key, value in res.items():
        if key in telemetria_pos_minigame:
            if key == "nivel_energia" or key == "nivel_combustivel":
                telemetria_pos_minigame[key] = max(0.0, telemetria_pos_minigame[key] + value)
            elif key == "pressao_tanques_percentual":
                telemetria_pos_minigame[key] = value # Sobrescreve com o novo valor
            elif key == "velocidade_comunicacao":
                telemetria_pos_minigame[key] = value # Sobrescreve com o novo valor
            elif key == "integridade_estrutural":
                telemetria_pos_minigame[key] = value # Sobrescreve com o novo valor
        elif key == "o2_pct_global": # O2 é uma variável global, não parte da telemetria
            global o2_pct_global
            o2_pct_global = max(0.0, o2_pct_global + value)

    return telemetria_pos_minigame

# ─────────────────────────────────────────────
#  4. ALGORITMO DE VERIFICAÇÃO GO/NO-GO DETALHADO
# ─────────────────────────────────────────────
def verificar_go_no_go(telemetria):
    relatorio_falhas = []

    if not (5.0 <= telemetria.get("temperatura_interna", 0) <= 40.0):
        relatorio_falhas.append(f"Temperatura Interna fora da faixa segura ({telemetria.get('temperatura_interna', 0)}°C).")

    if not (4.0 <= telemetria.get("temperatura_externa", 0) <= 35.0):
        relatorio_falhas.append(f"Temperatura Externa fora da faixa segura ({telemetria.get('temperatura_externa', 0)}°C).")

    if telemetria.get("integridade_estrutural", 0) != 0:
        relatorio_falhas.append("Integridade Estrutural comprometida.")

    if telemetria.get("nivel_energia", 0) != 100.0:
        relatorio_falhas.append(f"Nível de Energia abaixo de 100% ({telemetria.get('nivel_energia', 0)}%).")

    if not (98.0 <= telemetria.get("pressao_tanques_percentual", 0) <= 102.0):
        relatorio_falhas.append(f"Pressão dos Tanques fora da margem de segurança ({telemetria.get('pressao_tanques_percentual', 0)}% da nominal).")

    if telemetria.get("status_modulos", 0) != 0:
        relatorio_falhas.append("Módulos Críticos não operacionais.")

    if telemetria.get("velocidade_vento", 0) >= 50.0:
        relatorio_falhas.append(f"Velocidade do Vento acima do limite seguro ({telemetria.get('velocidade_vento', 0)} km/h).")

    if telemetria.get("umidade_relativa", 0) >= 80.0:
        relatorio_falhas.append(f"Umidade Relativa acima do limite seguro ({telemetria.get('umidade_relativa', 0)}%).")

    if telemetria.get("visibilidade", 0) <= 10.0:
        relatorio_falhas.append(f"Visibilidade abaixo do mínimo seguro ({telemetria.get('visibilidade', 0)} km).")

    if telemetria.get("nivel_combustivel", 0) != 100.0:
        relatorio_falhas.append(f"Nível de Combustível abaixo de 100% ({telemetria.get('nivel_combustivel', 0)}%).")

    if telemetria.get("velocidade_comunicacao", 0) <= 5.0:
        relatorio_falhas.append(f"Velocidade de Comunicação abaixo do mínimo seguro ({telemetria.get('velocidade_comunicacao', 0)} Mbps).")

    if not (40.0 <= telemetria.get("angulo_fase", 0) <= 48.0):
        relatorio_falhas.append(f"Ângulo de Fase fora da faixa ideal ({telemetria.get('angulo_fase', 0)}°). Janela de lançamento fechada.")

    analise_energia = realizar_analise_energetica(telemetria.get("nivel_energia", 0))
    if not analise_energia["seguro"]:
        relatorio_falhas.append(f"Análise Energética: Energia insuficiente para decolagem segura. Restante: {analise_energia['restante']:.1f} kWh.")

    if not relatorio_falhas:
        return "PRONTO PARA DECOLAR", []
    else:
        return "DECOLAGEM ABORTADA", relatorio_falhas

# ─────────────────────────────────────────────
#  5. EXECUÇÃO PRINCIPAL
# ─────────────────────────────────────────────
def simulacao_aurora_siger():

    nome_csv = carregar_arquivo_local()

    if not nome_csv:
        print("Nenhum arquivo enviado. Simulação encerrada.")
        return

    try:
        df = pd.read_csv(nome_csv)
    except Exception as e:
        print(f"Erro ao ler o arquivo: {e}")
        return

    global o2_pct_global
    o2_pct_global = 100.0

    random_row = df.sample(n=1).iloc[0]
    telemetria_atual = random_row.to_dict()

    print(f"\n{'='*30} DADOS DO DATASET CARREGADO {'='*30}")
    for k, v in telemetria_atual.items():
        print(f"  - {k}: {v}")

    status_inicial, falhas_iniciais = verificar_go_no_go(telemetria_atual)

    if status_inicial == "PRONTO PARA DECOLAR":
        print("\n STATUS INICIAL: TUDO OK! INICIANDO EVENTO ALEATÓRIO...")
        telemetria_final = iniciar_mini_game(telemetria_atual)

        status_pos, falhas_pos = verificar_go_no_go(telemetria_final)
        if o2_pct_global < 50: falhas_pos.append("Oxigênio Crítico!")

        if status_pos == "PRONTO PARA DECOLAR" and not falhas_pos:
            exibir_sucesso()
        else:
            exibir_falha(falhas_pos)
    else:
        exibir_falha(falhas_iniciais)

if __name__ == "__main__":

    simulacao_aurora_siger()

Por favor, faça o upload do arquivo 'dataset_telemetria_marte.csv'


Saving dataset_telemetria_marte.csv to dataset_telemetria_marte (20).csv
 Arquivo "dataset_telemetria_marte (20).csv" carregado com sucesso!

============================== DADOS DO DATASET CARREGADO ==============================
  - temperatura_interna: 15.01
  - temperatura_externa: 22.08
  - integridade_estrutural: 1
  - nivel_energia: 100.0
  - pressao_tanques_percentual: 85.33
  - status_modulos: 0
  - velocidade_vento: 13.68
  - umidade_relativa: 53.61
  - visibilidade: 38.42
  - nivel_combustivel: 100.0
  - velocidade_comunicacao: 8.29
  - angulo_fase: 0.8
  - status_final: DECOLAGEM ABORTADA

  MISSÃO ABORTADA: FALHAS DETECTADAS
  - Integridade Estrutural comprometida.
  - Pressão dos Tanques fora da margem de segurança (85.33% da nominal).
  - Ângulo de Fase fora da faixa ideal (0.8°). Janela de lançamento fechada.
  PROCEDIMENTOS DE EMERGÊNCIA ATIVADOS. CÁPSULA EJETADA.
  [FALHA NA JORNADA PARA MARTE]
